# CEL-Seq2 to 10x Chromium V2
Run with the `scPath` kernel and a CUDA GPU. This example uses the current original model unchanged. The source PBMC adaptive rule selects GAT and a batch from the post-QC cell count; training uses 500 steps and the original student stage. Set PYTHONHASHSEED=42 before launching Jupyter for stable set-based gene ordering.
Place the prepared eight input files under data/RNA/cel_seq2 and the human Reactome/KEGG GMTs under data/pathway/human. See README.md for provenance and outputs.

In [1]:
import os, sys
from pathlib import Path
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scpath/models.py').is_file())
sys.path.insert(0, str(PROJECT_ROOT))
import torch
assert torch.cuda.is_available(), 'CUDA is required'
device = torch.device(os.environ.get('SCPATH_DEVICE', 'cuda:0'))
print('GPU:', torch.cuda.get_device_name(device))


GPU: NVIDIA GeForce RTX 4090


## Prepare the paired data
The companion processor reads the original exports, applies the source QC, builds Reactome/KEGG graphs, and selects reference HVGs for both splits. No historical feature manifest or checkpoint is substituted.

In [2]:
from examples.cel_seq2.process_data import prepare
reference, query, pathways, output = prepare(PROJECT_ROOT)
print('Reference:', reference.shape, 'Query:', query.shape)


<active-conda-env>/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
<active-conda-env>/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
Original cell count: 463
<active-conda-env>/lib/python3.9/site-packages/scanpy/preprocessing/_simple.py:283: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["n_cells"] = number
Cell count after filtering: 463
Gene count after filtering: 9038
Cell count after filtering: 3098
Gene count after filtering: 14870
Number of shared genes: 9028

Processing pathway: Reactome
Gene-set size: 620 genes
./scpath/utils/pathway.py:49: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is d

## Train and distill
The adaptive batch formula and the fixed 500 steps follow platform.ipynb. high=True retains the original private-class logic; the student then refines rejected assignments. The MLP gate is the active original implementation.

In [3]:
from scpath.config import Config
from scpath.train import train
total = reference.n_obs + query.n_obs
smallest = min(reference.n_obs, query.n_obs)
gnn = 'gat' if total < 10000 else 'gcn' if total <= 50000 else 'sage'
batch = smallest // 2 if smallest < 1000 else 512 if smallest < 3000 else 1024
batch = min(batch, smallest)
config = Config(str(output) + '/', reference.n_vars, device, True, batch_size=batch, gnn_type=gnn, max_steps=500, K=len(query.obs['cell_type'].unique()), rate=0.05)
print('GNN:', gnn, 'Batch:', batch, 'Steps:', config.max_steps)
train(config, reference, query, pathways)


[easydl] tensorflow not available!
GNN: gat Batch: 231 Steps: 500
Shared classes: 9
Reference-only classes: 0
Query-only classes: 0
['B cell', 'CD14+ monocyte', 'CD16+ monocyte', 'CD4+ T cell', 'Cytotoxic T cell', 'Dendritic cell', 'Megakaryocyte', 'Natural killer cell', 'Plasmacytoid dendritic cell']
463 463
3098 3098
463 463
3098 3098
                     cell_type  label
0                       B cell      0
1               CD14+ monocyte      1
2               CD16+ monocyte      2
3                  CD4+ T cell      3
4             Cytotoxic T cell      4
5               Dendritic cell      5
6                Megakaryocyte      6
7          Natural killer cell      7
8  Plasmacytoid dendritic cell      8
{'source_classes': [0, 1, 2, 3, 4, 5, 6, 7, 8], 'target_private_classes': [], 'common_classes': [0, 1, 2, 3, 4, 5, 6, 7, 8], 'all_classes': [0, 1, 2, 3, 4, 5, 6, 7, 8]}
2
Evaluating on target domain: 100%|##########| 14/14 [00:00<00:00, 17.68it/s]

Evaluating on source domain: 100

## Inspect measured results
The original PBMC evaluator pools query-private subclasses into one private class before scoring. These are measurements from this run, not a guaranteed manuscript value.

In [4]:
import scanpy as sc
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
result = sc.read_h5ad(output / 'logs/test_results_distilled.h5ad')
truth = np.minimum(result.obs['labels'].astype(int).to_numpy(), result.obsm['cls_probs'].shape[1])
rows = []
for method, column in [('OT common', 'pred_labels_ot_common'), ('Final distilled', 'final_distill_preds')]:
    pred = result.obs[column].astype(int).to_numpy()
    rows.append({'Method': method, 'Accuracy': accuracy_score(truth, pred), 'Macro_F1': f1_score(truth, pred, average='macro', zero_division=0)})
metrics = pd.DataFrame(rows)
metrics.to_csv(output / 'metrics.csv', index=False)
print(metrics.to_string(index=False))
print(result.obs[['cell_type', 'labels', 'final_distill_preds']].head())


         Method  Accuracy  Macro_F1
      OT common  0.933505  0.765424
Final distilled  0.951259  0.746919
                                         cell_type labels  final_distill_preds
cells                                                                         
pbmc2_10X_V2_AGTGAGGTCTTGAGGT          CD4+ T cell      3                    3
pbmc2_10X_V2_AGTGGGAAGTTCGCAT               B cell      0                    0
pbmc2_10X_V2_AGTGGGACACACCGCA       CD14+ monocyte      1                    1
pbmc2_10X_V2_AGTGTCACACCAACCG               B cell      0                    0
pbmc2_10X_V2_AGTGTCAGTAGAGCTG  Natural killer cell      7                    7
